In [ ]:
#!pip install kramersmoyal
import pandas as pd
import numpy as np
from kramersmoyal import km

import os
import pickle
   

from utils.Data_cleaning import data_cleaning
from utils.Functions import data_filter, integrate_omega, KM_Coeff_1, KM_Coeff_2, daily_profile, power_mismatch, exp_decay, Euler_Maruyama, Increments, autocor 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pysindy as ps
from scipy.integrate import solve_ivp
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from pysindy.feature_library import FourierLibrary
from pysindy.feature_library import CustomLibrary
from pysindy.optimizers import SR3
from scipy.ndimage import gaussian_filter1d
#import sdeint

# Integrator keywords for solve_ivp
integrator_keywords = {}
integrator_keywords['rtol'] = 1e-12
integrator_keywords['method'] = 'LSODA'
integrator_keywords['atol'] = 1e-12

#### Choose the grid: (Balearic, Irish or Iceland)

In [ ]:
#grids = ['IS02','IRL01','ES_PM01']  
#grids = ['Balearic','Irish','Iceland' ]  
#grids = ['Iceland']  
#grids = ['Irish']
grids = ['Balearic']

In [ ]:
dict_grids = {'Balearic':'ES_PM01','Irish':'IRL01','Iceland':'IS02'}
#dict_freq_grids = {'Balearic':'f50_ES_PM','Irish':'f50_IRL01'}
dict_freq_grids = {'Balearic':'f50_ES_PM','Irish':'f50_IRL01','Iceland':'f50_IS' }

In [ ]:
models = ['model 1','model 2','model 3','model 4']

#freq_orig = data/(2*np.pi+50)
#increments_orig = Increments(data/(2*np.pi+50))

'''For calculations: use angular velocity omega = 2*pi*frequency '''
'''The bandwidth is chosen such that we receive a scmooth distribution'''

'''Choose the grid '''

'''Data analysis of the original time series'''
data_orig          = {i:[]for i in grids}



edges_1d     = {i:[]for i in grids}
drift_1d     = {i:[]for i in grids}
diffusion_1d = {i:[]for i in grids}
edges_2d     = {i:[]for i in grids}
kmc_2d       = {i:[]for i in grids}

for grid in grids:
    time_res = 1
    '''Choose the grid '''
    #raw=pd.read_csv('./Data/Frequency_data_%s.csv'%(grid), sep=',')  
    #freq = (raw[['Frequency']]/1000 +50).squeeze()
    #freq = data_cleaning(freq)
    
    raw=pd.read_csv('Data/%s.csv'%(dict_grids[grid]), sep=';')  
    freq = (raw[[dict_freq_grids[grid]]]/1000 +50).squeeze()
    freq = data_cleaning(freq)
    
    #print(freq)


    data_orig[grid].append(freq)
   

The empirical data is given by data_orig[grid] (grid = Irish, Iceland, Balearic.)
In the following the different models are calculated, i.e. the synthetic time series is given by "omega_synth_model_i" (i=1,2,3,4).
\
The default length of the calculated synthetic time series is 5 days (t_final=5).
\
The time step delta_t should be chosen smaller or equal than one. Here we use delta_t = 0.1.

In [ ]:
def extract_data (data_set, n=900):
    """
    Extract  'omega' and 'theta' from the given dataset.
    """
    # Check if the dataset is not empty
    if data_set:
        omega = data_set[0][0]
        theta = data_set[0][1]

        
        omega_extract = omega[:n]
        theta_extract = theta[:n]

        return omega_extract, theta_extract

In [ ]:
def integrate_data(data, dt=1):
    """
    Manually integrate the data using a simple Euler integration method.

    Parameters:
    data (array): The input data to be integrated.
    dt (float): The time step for integration.

    Returns:
    integrated_data: The integrated data.
    """
    
    integrated_data = np.zeros(len(data))
    for i in range(1, len(data)):
        integrated_data[i] = integrated_data[i-1] + data[i] * dt
    return integrated_data

### Model 2 case A, B, and C

In [ ]:
'''Model 2...'''
np.random.seed(12345678)
noise_data_model_2 = {i:[]for i in grids}
ideal_data_model_2 = {i:[]for i in grids}
time_data_model_2 = {i:[]for i in grids}
c_1_model2 = {i:[]for i in grids}
c_2_model2 = {i:[]for i in grids}
epsilon_model2 = {i:[]for i in grids}

'''adapt the parameter estimation to the particulat grids'''
for grid in grids:

    #raw=pd.read_csv('./Data/Frequency_data_%s.csv'%(grid), sep=',')
    #freq = (raw[['Frequency']]/1000 +50).squeeze()
    #freq = data_cleaning(freq)
    
    raw=pd.read_csv('Data/%s.csv'%(dict_grids[grid]), sep=';')  
    freq = (raw[[dict_freq_grids[grid]]]/1000 +50).squeeze()
    freq = data_cleaning(freq)
    
    data = (freq-50)*(2*np.pi)   #Use the angular velocity for the calcualltions

    trend = 1 #trend is boolean
    bw_drift = 0.1
    bw_diff = 0.1
    dist_drift = 500    #for large data set: dist_drift = 350 for Balearic
    dist_diff = 500
    if grid == 'Balearic':
        Delta_P = power_mismatch(data,avg_for_each_hour = False,dispatch=2,start_minute=0,end_minute=1/6,length_seconds_of_interval=5)
        dispatch = 1
    elif grid == 'Irish':
        Delta_P = power_mismatch(data_filter(data,sigma = 6),avg_for_each_hour = False,dispatch=1,start_minute=0,end_minute=1/6,length_seconds_of_interval=5)
        dispatch = 2
        #we use a filter for the power mismatch of the Iroish data because of regular outliers (every 60 seconds)
    elif grid == 'Iceland':
        Delta_P = 0
        dispatch = 0
        trend = 0 # Represents a no-existing trend as there is no power dispatch schedule
        
    c_1 = KM_Coeff_1(data - trend*data_filter(data),dim= 1,time_res = 1,bandwidth = bw_drift,dist = dist_drift, order = 1)
    c_2_decay = trend*exp_decay(data,time_res=1,size = 899)
    epsilon =   epsilon = KM_Coeff_2(data - trend*data_filter(data),dim = 1,time_res = 1,bandwidth = bw_diff,dist = dist_diff,multiplicative_noise = False)
    #epsilon = 0

    kmc,edges = km(data - trend * data_filter(data),powers = [0,1,2],bins = np.array([6000]),bw=bw_drift)
    edges_1d[grid] = edges[0]
    drift_1d[grid] = kmc[1]
    diffusion_1d[grid] = kmc[2] 
    c_1_model2[grid] = c_1
    c_2_model2[grid] = c_2_decay * c_1
    epsilon_model2[grid] = epsilon

    delta_t = 1 #time step for Euler-Maruyama
    
    # with noise there are 96 intervals of 15 minutes each in one day.
    omega_noise_model_2, theta_noise_model_2, p1 = Euler_Maruyama(data,c_1=c_1,c_2_decay=c_2_decay,Delta_P = Delta_P,
                                                              epsilon=epsilon,time_res = 1,dispatch = dispatch,
                                                              delta_t=delta_t,t_final=900*96*30,model=2,factor_daily_profile=0,sawtooth = False)
    # without noise, set epsilon = 0
    omega_ideal_model_2, theta_ideal_model_2, p2= Euler_Maruyama(data,c_1=c_1,c_2_decay=c_2_decay,Delta_P = Delta_P,
                                                              epsilon=0,time_res = 1,dispatch = dispatch,
                                                              delta_t=delta_t,t_final=900*96*30,model=2,factor_daily_profile=0,sawtooth = False)
    # with sawtooth = True
    omega_time_model_2, theta_time_model_2, p3= Euler_Maruyama(data,c_1=c_1,c_2_decay=c_2_decay,Delta_P = Delta_P,
                                                              epsilon=epsilon,time_res = 1,dispatch = dispatch,
                                                              delta_t=delta_t,t_final=900*96*30,model=2,factor_daily_profile=0,sawtooth = True)
    
    #freq_synth_model_2 = omega_synth_model_2/(2*np.pi) + 50

    noise_data_model_2[grid].append((omega_noise_model_2, theta_noise_model_2))
    ideal_data_model_2[grid].append((omega_ideal_model_2, theta_ideal_model_2))
    time_data_model_2[grid].append((omega_time_model_2, theta_time_model_2))


    

In [ ]:
Bal_noise = noise_data_model_2['Balearic']
Bal_ideal = ideal_data_model_2['Balearic']
Bal_time = time_data_model_2['Balearic']

In [ ]:
import pickle

# Save Bal_noise to a file
with open('Bal_noise.pkl', 'wb') as file:
    pickle.dump(Bal_noise, file)

# Save Bal_ideal to a file
with open('Bal_ideal.pkl', 'wb') as file:
    pickle.dump(Bal_ideal, file)
    
# Save Bal_time to a file
with open('Bal_time.pkl', 'wb') as file:
    pickle.dump(Bal_time, file)


In [ ]:
# Load Bal_noise from the file
with open('Bal_noise.pkl', 'rb') as file:
    Bal_noise = pickle.load(file)

# Load Bal_ideal from the file
with open('Bal_ideal.pkl', 'rb') as file:
    Bal_ideal = pickle.load(file)

# Load Bal_time from the file
with open('Bal_time.pkl', 'rb') as file:
    Bal_time = pickle.load(file)

In [ ]:
omega_Bal_noise, theta_Bal_noise = extract_data(Bal_noise,n=900*96*30)

In [ ]:
omega_Bal_ideal, theta_Bal_ideal = extract_data(Bal_ideal,n=900*96*30)

In [ ]:
omega_Bal_time, theta_Bal_time = extract_data(Bal_time,n=900*96*30)

In [ ]:
np.mean(p1)

In [ ]:
np.mean(p2)

In [ ]:
np.mean(p3)

In [ ]:
plt.figure(figsize=(20, 6))  # Adjust the size as needed

plt.plot(omega_Bal_noise[:900*4+1] ,label='Omega Bal Noise')
plt.plot(omega_Bal_ideal[:900*4+1],label='Omega Bal Ideal')
plt.plot(omega_Bal_time[:900*4+1],label='Omega Bal Ideal')

plt.legend()  # Show legend if multiple lines are plotted
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Comparison of Omega Bal Noise and Omega Bal Ideal')

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))  # Adjust the size as needed

plt.plot(theta_Bal_noise[:900*96*3], label='Theta Bal Noise')
plt.plot(theta_Bal_ideal[:900*96*3], label='Theta Bal Ideal')
plt.plot(theta_Bal_time[:900*96*3], label='Theta Bal Time')

plt.legend()  # Show legend if multiple lines are plotted
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Comparison of Theta Bal Noise and Theta Bal Ideal')

plt.show()

## Linear noise-free validation model

In [ ]:
omega_df = pd.DataFrame({'omega': omega_Bal_ideal})
theta_df = pd.DataFrame({'theta': theta_Bal_ideal})

# Combine omega and theta DataFrames into a single DataFrame
combined_df = pd.concat([omega_df, theta_df], axis=1)

# Define the new list to hold the chunked DataFrames
chunked_data = []

# Number of points in each chunk
chunk_size = 900*4

# Calculate the number of chunks needed
num_chunks = len(combined_df) // chunk_size

# Split the DataFrame into chunks
for j in range(num_chunks):
    # Extract the chunk
    chunk = combined_df.iloc[j * chunk_size:(j + 1) * chunk_size].copy()

    # Label the chunk with the region name
    chunk['Region'] = 'Balearic'

    # Append the chunk to the list
    chunked_data.append(chunk)

# If there are remaining data points
if len(combined_df) % chunk_size != 0:
    remaining_chunk = combined_df.iloc[num_chunks * chunk_size:].copy()
    
    # Label the remaining chunk with the region name
    remaining_chunk['Region'] = 'Balearic'
    
    # Append the remaining chunk to the list
    chunked_data.append(remaining_chunk)




In [ ]:
def noiseless_sindy_models(region_data, region_name):
    # Initialize an empty list for sindy_models
    sindy_models = []

    # Iterate over each chunk
    for chunk in region_data:
        # Integrate omega_filtered to get theta
        theta_chunk = chunk['theta'].values

        # Stack normalized theta and omega_filtered
        stacked_data_chunk = np.column_stack((theta_chunk, chunk['omega'].values))

        # Generate time values
        t_train_chunk = np.arange(0, len(stacked_data_chunk), 1)

        # Add time as a linear feature
        linear_time_feature_chunk = t_train_chunk.reshape(-1, 1)
        x_train_augmented_chunk = np.hstack([stacked_data_chunk, linear_time_feature_chunk])
        #x_train_augmented_chunk = np.hstack([stacked_data_chunk])

        # Feature names for SINDy (including a linear term for time)
        feature_names_chunk = ["theta", "omega", "time"]
        #feature_names_chunk = ["theta", "omega"]

        # Use Polynomial Library 
        polynomial_library_chunk = ps.PolynomialLibrary(degree=2)

        # Sparse regression optimizer
        sparse_regression_optimizer_chunk = ps.STLSQ(threshold=1e-10)

        # Create a SINDy model with the polynomial library
        model_chunk = ps.SINDy(feature_names=feature_names_chunk, 
                               feature_library=polynomial_library_chunk,
                               optimizer=sparse_regression_optimizer_chunk)

        # Fit the SINDy model using the chunk data
        model_chunk.fit(x_train_augmented_chunk, t=1)

        # Append the SINDy model to the list
        sindy_models.append(model_chunk)

        # Print the learned model for the current chunk
        print(f"Model for {region_name} - Chunk {len(sindy_models)}:")
        model_chunk.print(precision=7)
        print("Model Score:", model_chunk.score(x_train_augmented_chunk, t=1))
        print()

    # Initialize empty lists for coefficient matrices
    coefficients_matrices = []

    # Iterate over each SINDy model in sindy_models
    for model in sindy_models:
        # Get the coefficients from the model
        coefficients = model.coefficients()

        # Append the coefficients to the list
        coefficients_matrices.append(coefficients)

    # Convert the list of matrices to a 3D NumPy array
    coefficients_array = np.array(coefficients_matrices)

    # Calculate mean and standard deviation along the first axis (axis=0) which corresponds to chunks
    mean_coefficients = np.mean(coefficients_array, axis=0)
    std_coefficients = np.std(coefficients_array, axis=0)

    # Print mean_coefficients and std_coefficients 
    print(f"Mean Coefficients for {region_name}:")
    print(mean_coefficients)
    print(f"Standard Deviation of Coefficients Coefficients for {region_name}:")
    print(std_coefficients)
    
    return sindy_models



In [ ]:
def noiseless_sindy_models_abs(region_data, region_name):
    sindy_models = []
    coefficients_matrices = []  # Initialize outside the models loop

    for chunk in region_data:
        # Integrate omega_filtered to get theta
        theta_chunk = chunk['theta'].values

        # Stack normalized theta and omega_filtered
        stacked_data_chunk = np.column_stack((theta_chunk, chunk['omega'].values))

        # Generate time values
        t_train_chunk = np.arange(0, len(stacked_data_chunk), 1)

        # Add time as a linear feature
        linear_time_feature_chunk = t_train_chunk.reshape(-1, 1)
        x_train_augmented_chunk = np.hstack([stacked_data_chunk, linear_time_feature_chunk])
        #x_train_augmented_chunk = np.hstack([stacked_data_chunk])

        # Feature names for SINDy (including a linear term for time)
        feature_names_chunk = ["theta", "omega", "time"]
        #feature_names_chunk = ["theta", "omega"]

        # Use Polynomial Library 
        polynomial_library_chunk = ps.PolynomialLibrary(degree=2)

        # Sparse regression optimizer
        sparse_regression_optimizer_chunk = ps.STLSQ(threshold=1e-10)

        # Create a SINDy model with the polynomial library
        model_chunk = ps.SINDy(feature_names=feature_names_chunk, 
                               feature_library=polynomial_library_chunk,
                               optimizer=sparse_regression_optimizer_chunk)

        # Fit the SINDy model using the chunk data
        model_chunk.fit(x_train_augmented_chunk, t=1)

        # Append the SINDy model to the list
        sindy_models.append(model_chunk)

        # Print the learned model for the current chunk
        #print(f"Model for {region_name} - Chunk {len(sindy_models)}:")
        #model_chunk.print(precision=7)
        #print("Model Score:", model_chunk.score(x_train_augmented_chunk, t=1))
        #print()

    for model in sindy_models:
        coefficients = model.coefficients()
        feature_names = model.get_feature_names()

        # Handling the presence of specific features
        time_indices = [feature_names.index(f) for f in feature_names if 'time' in f]

        # Take the absolute value of specific coefficients
        for idx in time_indices:
            coefficients[:, idx] = np.abs(coefficients[:, idx])

        coefficients_matrices.append(coefficients)

    # Convert to a 3D NumPy array and calculate mean and standard deviation
    coefficients_array = np.array(coefficients_matrices)
    mean_coefficients = np.mean(coefficients_array, axis=0)
    std_coefficients = np.std(coefficients_array, axis=0)

    print(f"Mean Coefficients for {region_name}:")
    print(mean_coefficients)
    print(f"Standard Deviation of Coefficients for {region_name}:")
    print(std_coefficients)

    return sindy_models


In [ ]:
def noiseless_simulate_sindy_model(model, initial_conditions, time_points, title, omega_original):
    # Simulate the system using the provided model
    simulated_data = model.simulate(initial_conditions, time_points)

    # Extract simulated theta and omega
    simulated_theta = simulated_data[:, 0]
    simulated_omega = simulated_data[:, 1]

    # Plot the simulation results
    plt.figure(figsize=(10, 4))

    # Plot the original Omega with noise
    plt.plot(omega_original, label='Linear Noise-Free', alpha=0.7, color='#2b6a99')

    # Plot the simulated Omega
    plt.plot(simulated_omega, label='Simulated Omega', linestyle='--', linewidth=3, color='#f16c23')

    plt.title(f'Comparison of Linear Noise-Free and Simulated Omega - {title}', fontsize=16)
    plt.xlabel('Time', fontsize=16)
    plt.ylabel('Omega', fontsize=16)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    
    plt.legend(fontsize=16,frameon=False)
    plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    plt.show()


In [ ]:
balearic_ideal = chunked_data
ideal_abs_models = noiseless_sindy_models_abs(balearic_ideal, 'Balearic')

In [ ]:
#['1', 'theta', 'omega','time',
# 'theta^2','theta omega','theta time','omega^2',
# 'omega time','time^2']

In [ ]:
balearic_ideal = chunked_data
balearic_models = noiseless_sindy_models(balearic_ideal, 'Balearic')

In [ ]:
c_1_model2

In [ ]:
c_2_model2

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
n = 99 

# Extract initial conditions consistently
initial_conditions = [balearic_ideal[n]['theta'].values[0], 
                      balearic_ideal[n]['omega'].values[0],0]

# Define time points for simulation
total_time = len(balearic_ideal[n]['omega'])
time_points = np.arange(0, total_time, 1)

# Assuming omega_original is available
omega_original = balearic_ideal[n]['omega'].values  # Replace 'omega' with the actual column name

# Simulate and plot results for a specific chunk (e.g., chunk n+1)
noiseless_simulate_sindy_model(balearic_models[n], initial_conditions, time_points, f'Balearic - Chunk {n+1}', omega_original)


##  LINEAR VALIDATION MODEL INCLUDING NOISE


In [ ]:
import pandas as pd
from scipy.ndimage import gaussian_filter1d

def apply_gaussian_filter(data, chunk_size=900*4, sigma=60):
    # Create DataFrame
    data_df = pd.DataFrame({'omega': data})

    # Define the list to hold the filtered DataFrames
    filtered_data = []

    # Calculate the number of chunks needed
    num_chunks = len(data_df) // chunk_size

    # Split the DataFrame into chunks and apply the Gaussian filter to each chunk
    for j in range(num_chunks):
        # Extract the chunk
        chunk = data_df.iloc[j*chunk_size:(j+1)*chunk_size].copy()
        
        # Apply Gaussian filter to the 'omega' column
        chunk['omega_filtered'] = gaussian_filter1d(chunk['omega'], sigma=sigma)
        
        # Label the chunk with 'balearic'
        chunk['Region'] = 'balearic'
        
        # Append the filtered chunk to the list
        filtered_data.append(chunk)

    # If there are remaining data points
    if len(data_df) % chunk_size != 0:
        remaining_chunk = data_df.iloc[num_chunks*chunk_size:].copy()
        remaining_chunk['omega_filtered'] = gaussian_filter1d(remaining_chunk['omega'], sigma=sigma)
        
        # Label the remaining chunk with 'balearic'
        remaining_chunk['Region'] = 'balearic'
        
        # Append the remaining chunk to the list
        filtered_data.append(remaining_chunk)
    
    return filtered_data

In [ ]:
def calculate_sindy_models(region_data, region_name):
    # Initialize an empty list for sindy_models
    sindy_models = []

    # Iterate over each chunk
    for chunk in region_data:
        # Integrate omega_filtered to get theta
        theta_chunk = integrate_data(chunk['omega_filtered'].values)

        # Stack normalized theta and omega_filtered
        stacked_data_chunk = np.column_stack((theta_chunk, chunk['omega_filtered'].values))

        # Generate time values
        t_train_chunk = np.arange(0, len(stacked_data_chunk), 1)

        # Add time as a linear feature
        linear_time_feature_chunk = t_train_chunk.reshape(-1, 1)
        x_train_augmented_chunk = np.hstack([stacked_data_chunk, linear_time_feature_chunk])
        #x_train_augmented_chunk = np.hstack([stacked_data_chunk])

        # Feature names for SINDy (including a linear term for time)
        feature_names_chunk = ["theta", "omega", "time"]
        #feature_names_chunk = ["theta", "omega"]

        # Use Polynomial Library 
        polynomial_library_chunk = ps.PolynomialLibrary(degree=2)

        # Sparse regression optimizer
        sparse_regression_optimizer_chunk = ps.STLSQ(threshold=1e-10)

        # Create a SINDy model with the polynomial library
        model_chunk = ps.SINDy(feature_names=feature_names_chunk, 
                               feature_library=polynomial_library_chunk,
                               optimizer=sparse_regression_optimizer_chunk)

        # Fit the SINDy model using the chunk data
        model_chunk.fit(x_train_augmented_chunk, t=1)

        # Append the SINDy model to the list
        sindy_models.append(model_chunk)

        # Print the learned model for the current chunk
        print(f"Model for {region_name} - Chunk {len(sindy_models)}:")
        model_chunk.print(precision=7)
        print("Model Score:", model_chunk.score(x_train_augmented_chunk, t=1))
        print()

    # Initialize empty lists for coefficient matrices
    coefficients_matrices = []

    # Iterate over each SINDy model in sindy_models
    for model in sindy_models:
        # Get the coefficients from the model
        coefficients = model.coefficients()

        # Append the coefficients to the list
        coefficients_matrices.append(coefficients)

    # Convert the list of matrices to a 3D NumPy array
    coefficients_array = np.array(coefficients_matrices)

    # Calculate mean and standard deviation along the first axis (axis=0) which corresponds to chunks
    mean_coefficients = np.mean(coefficients_array, axis=0)
    std_coefficients = np.std(coefficients_array, axis=0)

    # Print or use mean_coefficients and std_coefficients as needed
    print(f"Mean Coefficients for {region_name}:")
    print(mean_coefficients)
    print(f"Standard Deviation of Coefficients Coefficients for {region_name}:")
    print(std_coefficients)
    
    return sindy_models

In [ ]:
def simulate_sindy_model(model, initial_conditions, time_points, title, omega_original, omega_filtered):
    # Simulate the system using the provided model
    simulated_data = model.simulate(initial_conditions, time_points)

    # Extract simulated theta and omega
    simulated_theta = simulated_data[:, 0]
    simulated_omega = simulated_data[:, 1]

    # Plot the simulation results
    plt.figure(figsize=(10, 6))

    # Plot the original Omega with noise
    plt.plot(omega_original, label='Original Omega with noise', alpha=0.7, color='#2b6a99')

    # Plot the Gaussian Filtered Omega
    plt.plot(omega_filtered, label='Gaussian Filtered Omega', linestyle='-', linewidth=2, color='#1b7c3d')

    # Plot the simulated Omega
    plt.plot(simulated_omega, label='Simulated Omega', linestyle='--', linewidth=3, color='#f16c23')

    plt.title(f'Comparison of Original, Filtered, and Simulated Omega - {title}')
    plt.xlabel('Time')
    plt.ylabel('Omega')
    plt.legend()
    plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    plt.show()

In [ ]:
def calculate_sindy_models_abs(region_data, region_name):
    # Initialize an empty list for sindy_models
    sindy_models = []

    # Iterate over each chunk
    for chunk in region_data:
        # Integrate omega_filtered to get theta
        theta_chunk = integrate_data(chunk['omega_filtered'].values)

        # Stack normalized theta and omega_filtered
        stacked_data_chunk = np.column_stack((theta_chunk, chunk['omega_filtered'].values))

        # Generate time values
        t_train_chunk = np.arange(0, len(stacked_data_chunk), 1)

        # Add time as a linear feature
        linear_time_feature_chunk = t_train_chunk.reshape(-1, 1)
        x_train_augmented_chunk = np.hstack([stacked_data_chunk, linear_time_feature_chunk])

        # Feature names for SINDy (including a linear term for time)
        feature_names_chunk = ["theta", "omega", "time"]

        # Use Polynomial Library 
        polynomial_library_chunk = ps.PolynomialLibrary(degree=2)

        # Sparse regression optimizer
        sparse_regression_optimizer_chunk = ps.STLSQ(threshold=1e-10)

        # Create a SINDy model with the polynomial library
        model_chunk = ps.SINDy(feature_names=feature_names_chunk, 
                               feature_library=polynomial_library_chunk,
                               optimizer=sparse_regression_optimizer_chunk)

        # Fit the SINDy model using the chunk data
        model_chunk.fit(x_train_augmented_chunk, t=1)

        # Append the SINDy model to the list
        sindy_models.append(model_chunk)

        # Print the learned model for the current chunk
        #print(f"Model for {region_name} - Chunk {len(sindy_models)}:")
        #model_chunk.print(precision=7)
        #print("Model Score:", model_chunk.score(x_train_augmented_chunk, t=1))
        #print()

    # Initialize empty lists for coefficient matrices
    coefficients_matrices = []
    
    for model in sindy_models:
        coefficients = model.coefficients()
        feature_names = model.get_feature_names()

        # Handling the presence of specific features
        time_indices = [feature_names.index(f) for f in feature_names if 'time' in f]

        # Take the absolute value of specific coefficients
        for idx in time_indices:
            coefficients[:, idx] = np.abs(coefficients[:, idx])

        coefficients_matrices.append(coefficients)

    # Convert to a 3D NumPy array and calculate mean and standard deviation
    coefficients_array = np.array(coefficients_matrices)
    mean_coefficients = np.mean(coefficients_array, axis=0)
    std_coefficients = np.std(coefficients_array, axis=0)

    print(f"Mean Coefficients for {region_name}:")
    print(mean_coefficients)
    print(f"Standard Deviation of Coefficients for {region_name}:")
    print(std_coefficients)

    return sindy_models


In [ ]:
omega_Bal_noise_filtered = apply_gaussian_filter(omega_Bal_noise, sigma=60)
balearic_data = [chunk for chunk in omega_Bal_noise_filtered if chunk['Region'].iloc[0] == 'balearic']
noise_models = calculate_sindy_models_abs(balearic_data, 'Balearic')

In [ ]:
#['1', 'theta', 'omega','time',
# 'theta^2','theta omega','theta time','omega^2',
# 'omega time','time^2']

In [ ]:
# for 'Balearic' region with noise
omega_Bal_noise_filtered = apply_gaussian_filter(omega_Bal_noise, sigma=60)
balearic_data = [chunk for chunk in omega_Bal_noise_filtered if chunk['Region'].iloc[0] == 'balearic']
balearic_models = calculate_sindy_models(balearic_data, 'Balearic')

In [ ]:
c_2_model2

In [ ]:
c_1_model2

In [ ]:
n=90
# Assume that 'omega_filtered' is the variable you want to integrate
initial_conditions = [integrate_data(balearic_data[n]['omega_filtered'].values)[0], 
                      balearic_data[n]['omega_filtered'].iloc[0],0]

# Define time points for simulation
total_time = len(balearic_data[1]['omega_filtered'])
time_points = np.arange(0, total_time, 1)

# Assuming omega_original and omega_filtered are available
omega_original = balearic_data[n]['omega'].values  # Replace 'omega' with the actual column name
omega_filtered = balearic_data[n]['omega_filtered'].values

# Simulate and plot results for a specific chunk (e.g., chunk 2)
simulate_sindy_model(balearic_models[n], initial_conditions, time_points, f'Balearic - Chunk {n+1}', omega_original, omega_filtered)


## LINEAR VALIDATION MODEL WITH NOISE AND TIME-DEPENDENT DRIVING

In [ ]:
# for 'Balearic' region with noise
omega_Bal_time_filtered = apply_gaussian_filter(omega_Bal_time, sigma=60)
balearic_data = [chunk for chunk in omega_Bal_time_filtered if chunk['Region'].iloc[0] == 'balearic']
time_models = calculate_sindy_models_abs(balearic_data, 'Balearic')

In [ ]:
balearic_models = calculate_sindy_models(balearic_data, 'Balearic')

In [ ]:
n=47
# Assume that 'omega_filtered' is the variable you want to integrate
initial_conditions = [integrate_data(balearic_data[n]['omega_filtered'].values)[0], 
                      balearic_data[n]['omega_filtered'].iloc[0],0]

# Define time points for simulation
total_time = len(balearic_data[1]['omega_filtered'])
time_points = np.arange(0, total_time, 1)

# Assuming omega_original and omega_filtered are available
omega_original = balearic_data[n]['omega'].values  # Replace 'omega' with the actual column name
omega_filtered = balearic_data[n]['omega_filtered'].values

# Simulate and plot results for a specific chunk (e.g., chunk 2)
simulate_sindy_model(balearic_models[n], initial_conditions, time_points, f'Balearic - Chunk {n+1}', omega_original, omega_filtered)
